# pyQuil Tutorial: Advanced Usage

*Condensed from the [pyQuil docs, v4.17](https://pyquil-docs.rigetti.com/en/stable/advanced_usage.html).*

> **Prerequisites:** `pip install pyquil`. Cells that run programs need the `qvm` and `quilc` servers running (the setup cell below starts them). Cells marked `# Requires QCS access` need a Rigetti Quantum Cloud Services account and will fail on a local-only setup.

## Setup

Use the **Python (Rigetti)** kernel (see the README for one-time kernel registration). Run the next cell once per session to start the `quilc` and `qvm` servers in the background (`!cmd &` doesn't work in Jupyter, so this uses `subprocess.Popen`).

In [ ]:
import subprocess
ACT = "source /ocean/projects/pscstaff/nnepal/Rigetti/.venv/bin/activate"
for cmd in ("qvm -S", "quilc -S -P"):
    name = cmd.split()[0]
    subprocess.Popen(["bash", "-c", f"{ACT} && exec {cmd}"],
                     stdout=open(f"{name}.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)

## pyQuil configuration

`QCSClient` manages connections to the compilation and execution components (quilc, QVM and QCS). APIs that accept a configuration object treat it as optional and load a default automatically. Override the defaults by creating your own `QCSClient` or by setting `QCS_SETTINGS_FILE_PATH` and `QCS_SECRETS_FILE_PATH`.

By default configuration is read from `$HOME/.qcs/settings.toml` and `$HOME/.qcs/secrets.toml`. Override the QVM and quilc URLs with `QCS_SETTINGS_APPLICATIONS_QVM_URL` and `QCS_SETTINGS_APPLICATIONS_QUILC_URL`.

Defaults when neither the settings file nor environment variables set them:

- QVM: `http://127.0.0.1:5000`
- quilc: `tcp://127.0.0.1:5555`

In [1]:
import os
# Uncomment to point pyQuil at servers on other hosts/ports (set before creating a QuantumComputer)
# os.environ["QCS_SETTINGS_APPLICATIONS_QVM_URL"] = "http://127.0.0.1:5000"
# os.environ["QCS_SETTINGS_APPLICATIONS_QUILC_URL"] = "tcp://127.0.0.1:5555"
print({k: v for k, v in os.environ.items() if k.startswith("QCS_")})

{}


## Multithreading

`QuantumComputer` objects are thread-safe and can be shared across threads. `Program` and `EncryptedProgram` are **not** thread-safe — `copy()` them before using them concurrently.

> The QVM processes requests in parallel; a QPU may process them sequentially or in parallel, depending on the qubits used.

If large batches time out on a QPU, raise `execution_timeout` (in seconds) in `get_qc()`.

**Recommendation:** use at least 2× parallelism to keep the QVM or QPU fully busy. Prefer thread pools; pyQuil has no asyncio API.

The docs example targets `Aspen-M-3` (requires QCS); here it runs on a local QVM:

In [ ]:
import warnings
from pyquil._deprecation import PyQuilDeprecationWarning
# Hide pyQuil 4.22's notices that the QVM is deprecated in favour of pyQuil v5
# (the QVM still works; see 0_Jupyter_Pykernel_from_Container_image.ipynb).
warnings.filterwarnings("ignore", category=PyQuilDeprecationWarning)

from multiprocessing.pool import ThreadPool

from pyquil import get_qc, Program
from pyquil.api import QCSClient

qc = get_qc("2q-qvm")  # docs: get_qc("Aspen-M-3")  # Requires QCS access


def run(program: Program):
    return qc.run(qc.compile(program)).get_register_map().get("ro")


programs = [
    Program(
        "DECLARE ro BIT",
        "RX(pi) 0",
        "MEASURE 0 ro",
    ).wrap_in_numshots_loop(10),
] * 20

with ThreadPool(5) as pool:
    results = pool.map(run, programs)

for i, result in enumerate(results):
    print(f"Results for program {i}:\n{result}\n")

## Alternative QPU endpoints

Rigetti QCS supports alternative endpoints for QPU access — useful, for example, for mock endpoints that simulate execution without an active reservation. Pass an `endpoint_id` wherever a `quantum_processor_id` goes. Compilation behaves the same as with the default endpoint, but execution goes through the alternate service, so don't treat results as meaningful.

In [ ]:
# Requires QCS access
from pyquil import get_qc
from pyquil.api import QPU

# Option 1
qc = get_qc("Aspen-M-3", endpoint_id="my_endpoint")

# Option 2
qam = QPU(quantum_processor_id="Aspen-M-3", endpoint_id="my_endpoint")

## Using libquil for quilc and QVM

> This feature is experimental and may not work on all platforms.

[libquil](https://github.com/rigetti/libquil) provides quilc and QVM as a library, so no server processes are needed. After installing libquil, install a compatible `qcs-sdk-python`:

```text
poetry run pip install --config-settings=build-args='--features libquil' qcs-sdk-python --force-reinstall --no-binary qcs-sdk-python
```

Check whether libquil is available:

In [ ]:
from pyquil.diagnostics import get_report
print(get_report())

The end of the report should look like:

```text
libquil:
    available: true
    quilc version: 1.27.0
    qvm version: 1.17.2 (077ba23)
```

If `available: true`, pass libquil clients to `get_qc()`:

In [ ]:
# Only works if the report above shows `available: true`
from pyquil import get_qc
from qcs_sdk.compiler.quilc import QuilcClient
from qcs_sdk.qvm import QVMClient

qc = get_qc("8q-qvm", quilc_client=QuilcClient.new_libquil(), qvm_client=QVMClient.new_libquil())

## Using qubit placeholders

> `QubitPlaceholder` gives you much the same thing as writing a function that returns a `Program` and takes qubit indices as arguments.

Normally pyQuil identifies qubits with integers:

In [ ]:
from pyquil import Program
from pyquil.gates import CNOT, H
print(Program(H(0), CNOT(0, 1)))

On near-term QPUs the specific physical qubits matter. To write a program once and run it on different qubits, use `QubitPlaceholder`s:

In [ ]:
from pyquil.quilatom import QubitPlaceholder
q0 = QubitPlaceholder()
q1 = QubitPlaceholder()
p = Program(H(q0), CNOT(q0, q1))
print(p)

### Addressing qubits

A program with unresolved placeholders can't run — they must be mapped to physical qubits first. `address_qubits()` maps them to 0…N by default, skipping indices already in use:

In [ ]:
from pyquil.quil import address_qubits
print(address_qubits(p))

Or give an explicit mapping:

In [ ]:
print(address_qubits(p, qubit_mapping={
    q0: 14,
    q1: 19,
}))

Alternatively, `resolve_placeholders_with_custom_resolvers()` takes functions that receive a placeholder and return a fixed value, or `None` to leave it unresolved:

In [ ]:
from typing import Optional
from pyquil import get_qc

q0 = QubitPlaceholder()
q1 = QubitPlaceholder()
p = Program(H(q0), CNOT(q0, q1))
qc = get_qc("2q-qvm")


def qubit_resolver(placeholder: QubitPlaceholder) -> Optional[int]:
    if placeholder == q0:
        return 0
    if placeholder == q1:
        return None


p.resolve_placeholders_with_custom_resolvers(qubit_resolver=qubit_resolver)
print(p)

### Requesting a register of qubit placeholders

`QubitPlaceholder.register()` creates a whole register at once:

In [ ]:
qbyte = QubitPlaceholder.register(8)
p_evens = Program(H(q) for q in qbyte)
print(address_qubits(p_evens, {q: i*2 for i, q in enumerate(qbyte)}))

## Classical control flow

> Dynamic control flow can have unexpected effects on readout data. See *Accessing raw execution data* in the docs.

### While loops

1. Declare a register to act as the loop's boolean test.
2. Initialize it to `1` (the loop preamble).
3. Write the loop body as a separate `Program`.
4. Use `while_do()` to add the control flow.
5. Call `resolve_label_placeholders()` to resolve the label placeholders it inserted.

In [ ]:
from pyquil import Program
from pyquil.gates import *

# Initialize the Program and declare a 1 bit memory space for our boolean flag
outer_loop = Program()
flag_register = outer_loop.declare('flag_register', 'BIT')

# Set the initial flag value to 1
outer_loop += MOVE(flag_register, 1)

# Define the body of the loop with a new Program
inner_loop = Program()
inner_loop += Program(X(0), H(0))
inner_loop += MEASURE(0, flag_register)

# Run inner_loop in a loop until flag_register is 0
outer_loop.while_do(flag_register, inner_loop)
outer_loop.resolve_label_placeholders()

print(outer_loop)

Classical instructions you can use on registers include:

- `NOT` — flips a classical bit
- `AND` — operates on two classical bits
- `IOR` — operates on two classical bits
- `MOVE` — moves a classical bit's value to another address
- `EXCHANGE` — swaps two classical bits' values

> You can build "numshots" loops in pure Quil this way; see `with_loop()` and *Build a fixed-count loop with Quil* in the docs.

### If, then

`if_then()` takes a test register and a program for each branch:

In [ ]:
# Declare our memory spaces
branching_prog = Program()
ro = branching_prog.declare('ro', 'BIT')
test_register = branching_prog.declare('test_register', 'BIT')

# Construct each branch of our if-statement. We can have empty branches
# simply by having empty programs.
then_branch = Program(X(0))
else_branch = Program()

# Construct our program so that the result in test_register is equally likely to be a 0 or 1
branching_prog += H(1)
branching_prog += MEASURE(1, test_register)

# Add the conditional branching
branching_prog.if_then(test_register, then_branch, else_branch)

# Measure qubit 0 into our readout register
branching_prog += MEASURE(0, ro)
branching_prog.resolve_label_placeholders()

print(branching_prog)

Run it several times and look at the test register:

In [ ]:
from pyquil import get_qc

qc = get_qc("2q-qvm")
branching_prog.wrap_in_numshots_loop(10)
result = qc.run(branching_prog)
print(result.get_register_map()['test_register'])

### Sentinel-based loop

Combining loops and conditionals gives a loop that repeats until a condition is met. Here the body prepares and measures a Bell state, and we repeat until both qubits read `0`.

In [ ]:
# Import some types we'll use
from typing import Optional, Tuple

# We'll use numpy to help us validate our results
import numpy as np

# We'll need to create a program and define an executor
from pyquil import Program, get_qc
# We'll use these gates in our program
from pyquil.gates import CNOT, H, X
# We'll also need the help of a few control flow instructions
from pyquil.quilbase import Halt, Qubit, MemoryReference, JumpTarget, Jump
from pyquil.quilatom import Label

#### The program body

The repeating body prepares a Bell state and measures both qubits into memory:

In [ ]:
def body(qubits: Tuple[Qubit, Qubit], measures: MemoryReference) -> Program:
    """Constructs a bell state between the given qubit and measures them into the given memory reference."""
    program = Program(H(qubits[0]), CNOT(*qubits))
    program.measure_all(*zip(qubits, measures))
    return program

#### Resetting state

If either qubit measured `1`, flip it back with an `X`:

In [ ]:
def reset_bell_state(qubits: Tuple[Qubit, Qubit], measures: MemoryReference) -> Program:
    """Resets the state of the qubits if either of them measured to 1."""
    program = Program()
    program.if_then(measures[0], Program(X(qubits[0])))
    program.if_then(measures[1], Program(X(qubits[1])))
    return program

#### Enforcing a sentinel condition

End the program (with `Halt`) if the memory reference is `0`; otherwise reset and jump back:

In [ ]:
def enforce_sentinel(mem_ref: MemoryReference, else_program: Program) -> Program:
    """Ends the program if mem_ref is 0, otherwise executes else_program."""
    program = Program()
    # We use the `if_then` method here to help us construct our branch. As described above,
    # `if_then` takes a memory reference and two programs. It constructs a branch that
    # runs the first program if `mem_ref` is 1, otherwise it runs the second program.
    # Since we want to end the program if `mem_ref` is 0, we pass in our HALTing
    # program as the second program and the alternative branch as the first.
    program.if_then(mem_ref, else_program, Program(Halt()))
    return program

#### Putting it all together

In [ ]:
def sentinel_program(qubits: Tuple[Qubit, Qubit]) -> Program:
    # Create a label to reference the start of the program
    start_label = Label("start-loop")

    # Use the label to create a jump target at the beginning of the program
    program = Program(JumpTarget(start_label))

    # Declare a register to measure the qubits into
    measures = program.declare("measures", "BIT", 2)

    # Add the loop body to our program
    program += body(qubits, measures)

    # If the sentinel condition isn't met, then we:
    reset = Program(
        # Reset the state of our qubits
        reset_bell_state(qubits, measures),
        # Jump back to the start of the program
        Jump(start_label)
    )

    # Finally, if both Qubits measured to 0 (our sentinel), then we want to end the program
    # Otherwise, we try again.
    program += enforce_sentinel(measures[0], reset)

    # We used pyQuil to construct some of the branches for us, those methods use label placeholders
    # to avoid conflicts with existing labels in the program, so we resolve those placeholders here.
    program.resolve_label_placeholders()
    return program

#### Testing our program

Run 1000 shots on a QVM and check that the `measures` register contains only 0s:

In [ ]:
qubits = (Qubit(0), Qubit(1))

qc = get_qc("2q-qvm")
program = sentinel_program(qubits)
program.wrap_in_numshots_loop(1000)
print(program.out())
results = qc.run(program)
measures = results.get_register_map()["measures"]

assert np.all(measures == 0)
print("All shots ended in the sentinel state 00")

## Pauli operator algebra

`PauliTerm` and `PauliSum` represent sums of Pauli products such as

$$\sigma = \tfrac{1}{2}I - \tfrac{3}{4}X_0Y_1Z_3 + (5-2i)Z_1X_2$$

In [ ]:
from pyquil.paulis import ID, sX, sY, sZ

# Pauli term takes an operator "X", "Y", "Z", or "I"; a qubit to act on, and
# an optional coefficient.
a = 0.5 * ID()
b = -0.75 * sX(0) * sY(1) * sZ(3)
c = (5-2j) * sZ(1) * sX(2)

# Construct a sum of Pauli terms.
sigma = a + b + c
print(f"sigma = {sigma}")

Their main use is building exponentials $\exp[-i\beta\sigma]$ as parameterized Quil programs. There are also utilities for finding commuting subgroups and for Suzuki–Trotter approximations up to fourth order. Arithmetic simplifies automatically:

In [ ]:
from pyquil.paulis import exponential_map

sigma_cubed = sigma * sigma * sigma
print(f"Simplified: {sigma_cubed}\n")

# Produce Quil code to compute exp[iX]
H = -1.0 * sX(0)
print(f"Quil to compute exp[iX] on qubit 0:\n"
      f"{exponential_map(H)(1.0)}")

`exponential_map()` returns a function, so you can supply the multiplicative constant later — common in variational algorithms. Compute $\exp[-i\alpha H]$ for several α:

In [ ]:
expH = exponential_map(H)
print(f"0:\n{expH(0.0)}\n")
print(f"1:\n{expH(1.0)}\n")
print(f"2:\n{expH(2.0)}")

`exponential_map` also works with parametric compilation — the angle becomes a declared `REAL` parameter:

In [ ]:
from pyquil import Program

ham = sZ(0) * sZ(1)
prog = Program()
theta = prog.declare('theta', 'REAL')
prog += exponential_map(ham)(theta)
print(prog)